# DSPy-Guided Synthetic Image Generation with Pruna Optimization

## Overview

This cookbook demonstrates how to build a closed-loop synthetic image generation pipeline that combines:

1. Inference Optimization with Pruna
Quantization, compilation, and runtime optimization for faster image generation.

2. Prompt Refinement with DSPy
Structured prompt generation and iterative improvement using programmable LLM pipelines.

3. Vision-Language Evaluation
Automated image quality assessment using a SmolVLM-based evaluator.

4.Ranking and Dataset Curation
Top-K filtering to retain only the highest-quality synthetic samples.

The final pipeline produces a curated synthetic dataset by iteratively generating, evaluating and refining image outputs.


## 1. Installation & Setup

### Prerequisites

This tutorial uses:

Pruna for inference optimization and model compression, DSPy for structured prompt orchestration, Transformers for VLM-based evaluation
, Datasets for synthetic dataset construction and Pillow for image processing utilities

In [ ]:
!pip install pruna dspy-ai transformers pillow datasets accelerate

In [ ]:
import os
import json
import logging
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import torch
from PIL import Image
from tqdm import tqdm

# DSPy
import dspy
from dspy import settings

# Pruna
from pruna import smash, SmashConfig, PrunaModel

# Hugging Face
import datasets
from huggingface_hub import notebook_login
from diffusers import Flux2Pipeline

logger = logging.getLogger(__name__)

## Authentication

Authenticate with Hugging Face to download gated models and upload generated datasets.

In [ ]:
notebook_login()

### Device & Cache Setup

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {device}")

In [ ]:
OUTPUT_DIR = Path("./flux_dataset_output")
OUTPUT_DIR.mkdir(exist_ok=True)

IMAGES_DIR = OUTPUT_DIR / "images"
IMAGES_DIR.mkdir(exist_ok=True)

METADATA_PATH = OUTPUT_DIR / "metadata.json"

## Part A: Optimize FLUX.2-klein-4B with Pruna

Before generating synthetic images, we first optimize the base diffusion pipeline using Pruna.

Our optimization strategy focuses on:

INT8 transformer quantization to reduce memory usage
Torch compilation to improve kernel execution efficiency
preserving generation quality for downstream dataset curation

Rather than aggressively minimizing VRAM, the configuration below prioritizes a balance between:

inference speed
memory efficiency
visual fidelity

### A1: Load Base Model

In [ ]:
logger.info("Loading FLUX.2-klein-4B...")

pipe = Flux2Pipeline.from_pretrained(
    "black-forest-labs/FLUX.2-klein-4B"
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

logger.info("✓ Base model loaded")

### A2: Define Pruna SmashConfig

In [ ]:
logger.info("Creating SmashConfig...")

smash_config = SmashConfig({

    # ------------------------------------------------------------------
    # INT8 Quantization
    # ------------------------------------------------------------------
    "diffusers_int8": {
        "diffusers_int8_weight_bits": 8,

        # FP4 quantization provides a good balance
        # between compression and image quality.
        "diffusers_int8_quant_type": "fp4",

        # Double quantization is unnecessary
        # for a 4B-scale diffusion model.
        "diffusers_int8_double_quant": False,

        # Keep weights on GPU for lower latency.
        "diffusers_int8_enable_fp32_cpu_offload": False,

        # Quantize transformer-heavy components only.
        "diffusers_int8_target_modules": {
            "include": [
                "*transformer*",
                "*attn*"
            ],
            "exclude": [
                "*norm*",
                "*embed*",
                "*pe_*"
            ]
        },
    },

    # ------------------------------------------------------------------
    # Torch Compile
    # ------------------------------------------------------------------
    "torch_compile": {
        "torch_compile_mode": "reduce-overhead",
        "torch_compile_backend": "inductor",

        # Avoid overly rigid graph specialization.
        "torch_compile_fullgraph": False,

        # Static shapes improve compilation stability.
        "torch_compile_dynamic": False,

        # Improves portability across environments.
        "torch_compile_make_portable": True,

        "torch_compile_target": "module_list",
    },
})

logger.info("✓ SmashConfig created")

### A3: Apply Pruna Smash

In [ ]:
logger.info("Applying Pruna optimizations...")

flux = smash(
    pipe,
    smash_config
)

logger.info("✓ Model optimization complete")

### A4: Save Optimized Model

In [ ]:
SMASHED_MODEL_PATH = OUTPUT_DIR / "flux_klein_4b_smashed"

logger.info(f"Saving optimized model to {SMASHED_MODEL_PATH}")

flux.save_pretrained(
    str(SMASHED_MODEL_PATH)
)

logger.info("✓ Optimized model saved")

## Part B: Configure VLM Grader (SmolVLM + DSPy)

To automatically curate generated images, we use a lightweight Vision-Language Model (VLM) as an evaluator.

The evaluator is responsible for:

scoring image quality
validating prompt alignment
filtering low-quality generations
enabling closed-loop dataset refinement

For this tutorial, we use SmolVLM-Instruct as the grading backend because it is lightweight enough for repeated inference while still capable of multimodal reasoning.

### B1: Initialize SmolVLM via DSPy

In [ ]:
logger.info("Initializing SmolVLM evaluator...")

vlm = dspy.HFModel(
    model="HuggingFaceTB/SmolVLM-Instruct",

    # Indicates that the backend accepts image inputs.
    hf_type="vision_model",

    # Deterministic decoding improves grading consistency.
    temperature=0.3,
    top_p=1.0,

    # Evaluation responses are intentionally short.
    max_tokens=256,
)

# Configure DSPy globally.
settings.configure(
    lm=vlm
)

logger.info("✓ DSPy configured with SmolVLM")

## Part C: Define Grading Schema

### C1: Image Grade Signature

In [ ]:
class ImageGrade(dspy.Signature):
    """
    Structured grading signature for VLM evaluator.
    
    Dimensions:
    - prompt_adherence (0-10): How well does image match the prompt?
    - aesthetic_quality (0-10): Composition, colors, clarity
    - text_correctness (0-10): If prompt mentions text, is it rendered correctly?
    - brand_alignment (0-10): Does it feel professional/on-brand?
    """
    
    prompt: str = dspy.InputField(
        desc="The text prompt used to generate the image"
    )
    image: object = dspy.InputField(
        desc="PIL Image object to evaluate"
    )
    
    prompt_adherence: float = dspy.OutputField(
        desc="Score 0-10: Alignment between prompt and generated image"
    )
    aesthetic_quality: float = dspy.OutputField(
        desc="Score 0-10: Visual quality, composition, clarity"
    )
    text_correctness: float = dspy.OutputField(
        desc="Score 0-10: Correctness of text rendering (if any)"
    )
    brand_alignment: float = dspy.OutputField(
        desc="Score 0-10: Professional appearance, brand fitness"
    )

### C2: VLM Grader Module

In [ ]:
class VLMGrader(dspy.Module):
    """
    Module that uses SmolVLM to grade images.
    """
    
    def __init__(self):
        super().__init__()
        self.predictor = dspy.Predict(ImageGrade)
    
    def forward(self, prompt: str, image: Image.Image) -> Dict[str, float]:
        """
        Grade a single image.
        
        Args:
            prompt: The original generation prompt
            image: PIL Image object
        
        Returns:
            Dictionary with scores for each dimension
        """
        prediction = self.predictor(prompt=prompt, image=image)
        
        return {
            "prompt_adherence": float(prediction.prompt_adherence),
            "aesthetic_quality": float(prediction.aesthetic_quality),
            "text_correctness": float(prediction.text_correctness),
            "brand_alignment": float(prediction.brand_alignment),
        }

# Instantiate grader
vlm_grader = VLMGrader()

### C3: Composite Scoring Metric

In [ ]:
class ScoringWeights:
    """Weights for composite score calculation"""
    PROMPT_ADHERENCE = 0.40
    AESTHETIC_QUALITY = 0.30
    TEXT_CORRECTNESS = 0.20
    BRAND_ALIGNMENT = 0.10

def compute_composite_score(scores: Dict[str, float]) -> float:
    """
    Compute weighted composite score.
    
    Weights emphasize prompt adherence and aesthetics,
    as these drive dataset quality.
    """
    return (
        ScoringWeights.PROMPT_ADHERENCE * scores.get("prompt_adherence", 0) +
        ScoringWeights.AESTHETIC_QUALITY * scores.get("aesthetic_quality", 0) +
        ScoringWeights.TEXT_CORRECTNESS * scores.get("text_correctness", 0) +
        ScoringWeights.BRAND_ALIGNMENT * scores.get("brand_alignment", 0)
    )

# Example
example_scores = {
    "prompt_adherence": 9.0,
    "aesthetic_quality": 8.5,
    "text_correctness": 7.0,
    "brand_alignment": 8.0,
}
print(f"Composite score: {compute_composite_score(example_scores):.2f}")

## Part D: Dataset Generation Loop

### D1: Define Controlled Subjects

In [ ]:
# These are your dataset subjects
# Keep this curated and small
SUBJECTS = [
    "Minimalist coffee brand poster with clean typography and steam",
    "Modern eco-friendly packaging design for luxury chocolate",
    "Contemporary tech startup office entrance with glass and wood",
    "Artisanal bakery storefront with warm lighting and fresh bread display",
    "Sustainable fashion brand lookbook with neutral tones and natural fabrics",
    "Boutique hotel lobby with marble and soft ambient lighting",
    "Organic skincare product flat lay with natural ingredients",
    "Furniture showroom displaying minimalist wooden pieces",
    "Plant-based restaurant menu photography with vibrant ingredients",
    "Luxury watch brand catalog shot with professional photography",
]

# For quick testing, use subset
SUBJECTS_TEST = SUBJECTS[:3]

logger.info(f"Using {len(SUBJECTS)} subjects for full run")
logger.info(f"Using {len(SUBJECTS_TEST)} subjects for test run")

### D2: Prompt Optimization with DSPy

In [ ]:
class PromptOptimizer(dspy.Module):
    """
    Uses DSPy to optimize base subject into detailed generation prompt.
    
    This can be simple (concatenation + templates) or
    complex (few-shot with examples, iterative refinement).
    
    For this tutorial: simple template-based approach.
    """
    
    def __init__(self):
        super().__init__()
        self.refine = dspy.ChainOfThought("subject -> optimized_prompt")
    
    def forward(self, subject: str) -> str:
        """
        Optimize a subject into a detailed, generation-ready prompt.
        
        Args:
            subject: High-level subject (e.g., "Minimalist coffee brand poster")
        
        Returns:
            Detailed prompt optimized for FLUX generation
        """
        # For this tutorial, use deterministic enrichment
        # In production, you could use dspy.ChainOfThought + LLM
        
        enrichments = {
            "style": "professional, high-quality product photography",
            "lighting": "soft natural lighting, studio-grade",
            "details": "sharp focus, vibrant colors, clean composition",
            "format": "4K, magazine-quality, modern aesthetic"
        }
        
        optimized = f"{subject}, {enrichments['style']}, {enrichments['lighting']}, {enrichments['details']}, {enrichments['format']}"
        
        return optimized

prompt_optimizer = PromptOptimizer()

### D3: Single Subject Generation Loop

In [ ]:
def generate_candidates_for_subject(
    subject: str,
    num_candidates: int = 4,
    max_retries: int = 2
) -> List[Tuple[Image.Image, Dict[str, float], str]]:
    """
    For a single subject:
    1. Optimize prompt
    2. Generate N candidates
    3. Grade each
    4. Return (image, scores, prompt) tuples
    
    Args:
        subject: Subject string
        num_candidates: Number of images to generate per subject
        max_retries: Retries on generation failure
    
    Returns:
        List of (image, scores_dict, prompt) tuples
    """
    logger.info(f"\n{'='*60}")
    logger.info(f"Subject: {subject}")
    logger.info(f"{'='*60}")
    
    # Step 1: Optimize prompt
    optimized_prompt = prompt_optimizer(subject=subject)
    logger.info(f"Optimized prompt: {optimized_prompt}")
    
    candidates = []
    
    # Step 2: Generate candidates
    for i in range(num_candidates):
        logger.info(f"  Generating candidate {i+1}/{num_candidates}...")
        
        retry_count = 0
        while retry_count < max_retries:
            try:
                # Generate image
                image = flux_smashed(
                    prompt=optimized_prompt,
                    num_inference_steps=20,  # Reduced for smashed model
                    guidance_scale=3.5,
                    height=768,
                    width=768,
                    seed=42 + i  # Deterministic for reproducibility
                ).images[0]
                
                break
            except Exception as e:
                retry_count += 1
                logger.warning(f"    Generation failed (attempt {retry_count}): {e}")
                if retry_count >= max_retries:
                    logger.error(f"    Skipping candidate after {max_retries} retries")
                    image = None
        
        if image is None:
            continue
        
        # Step 3: Grade image
        logger.info(f"    Grading image...")
        try:
            scores = vlm_grader(prompt=optimized_prompt, image=image)
            composite_score = compute_composite_score(scores)
            
            scores["composite"] = composite_score
            logger.info(f"    Composite score: {composite_score:.2f}/10.0")
            logger.info(f"      - Prompt adherence: {scores['prompt_adherence']:.1f}")
            logger.info(f"      - Aesthetic: {scores['aesthetic_quality']:.1f}")
            logger.info(f"      - Text: {scores['text_correctness']:.1f}")
            logger.info(f"      - Brand: {scores['brand_alignment']:.1f}")
            
            candidates.append((image, scores, optimized_prompt))
        
        except Exception as e:
            logger.error(f"    Grading failed: {e}")
            continue
    
    return candidates

### D4: Full Dataset Generation Pipeline

In [ ]:
def generate_full_dataset(
    subjects: List[str],
    candidates_per_subject: int = 4,
    keep_per_subject: int = 2,
    output_dir: Path = OUTPUT_DIR
) -> List[Dict]:
    """
    Full pipeline:
    1. For each subject:
       - Generate N candidates
       - Grade each
       - Keep top K
    2. Save images and metadata
    
    Args:
        subjects: List of subject strings
        candidates_per_subject: Number to generate per subject
        keep_per_subject: Number to keep per subject (top-k)
        output_dir: Where to save results
    
    Returns:
        List of dataset records (metadata)
    """
    logger.info(f"\n{'#'*60}")
    logger.info("STARTING DATASET GENERATION PIPELINE")
    logger.info(f"{'#'*60}")
    logger.info(f"Subjects: {len(subjects)}")
    logger.info(f"Candidates per subject: {candidates_per_subject}")
    logger.info(f"Keep per subject: {keep_per_subject}")
    logger.info(f"Expected output: {len(subjects) * keep_per_subject} images")
    
    all_records = []
    image_counter = 0
    
    for subject_idx, subject in enumerate(subjects, 1):
        logger.info(f"\n[{subject_idx}/{len(subjects)}]")
        
        # Generate candidates
        candidates = generate_candidates_for_subject(
            subject=subject,
            num_candidates=candidates_per_subject
        )
        
        if not candidates:
            logger.warning(f"No valid candidates for subject: {subject}")
            continue
        
        # Sort by composite score (descending)
        candidates.sort(key=lambda x: x[1]["composite"], reverse=True)
        
        # Keep top K
        selected = candidates[:keep_per_subject]
        logger.info(f"Selected {len(selected)} / {len(candidates)} candidates")
        
        # Save and record
        for rank, (image, scores, optimized_prompt) in enumerate(selected, 1):
            image_id = f"subject_{subject_idx:02d}_rank_{rank}"
            image_path = IMAGES_DIR / f"{image_id}.png"
            
            # Save image
            image.save(str(image_path))
            logger.info(f"  Saved: {image_path.name}")
            
            # Create record
            record = {
                "image_id": image_id,
                "subject": subject,
                "original_subject": subject,
                "optimized_prompt": optimized_prompt,
                "image_path": str(image_path.relative_to(OUTPUT_DIR)),
                "scores": {
                    "prompt_adherence": float(scores["prompt_adherence"]),
                    "aesthetic_quality": float(scores["aesthetic_quality"]),
                    "text_correctness": float(scores["text_correctness"]),
                    "brand_alignment": float(scores["brand_alignment"]),
                    "composite": float(scores["composite"]),
                },
                "rank_in_subject": rank,
            }
            
            all_records.append(record)
            image_counter += 1
    
    logger.info(f"\n{'#'*60}")
    logger.info(f"✓ PIPELINE COMPLETE")
    logger.info(f"Total images generated: {image_counter}")
    logger.info(f"{'#'*60}\n")
    
    return all_records

### D5: Run Generation (Full Run)

In [ ]:
# Run the full pipeline
dataset_records = generate_full_dataset(
    subjects=SUBJECTS_TEST,  # Use test subset for demo
    candidates_per_subject=4,
    keep_per_subject=2
)

# Save metadata
with open(METADATA_PATH, 'w') as f:
    json.dump(dataset_records, f, indent=2)

logger.info(f"Metadata saved to {METADATA_PATH}")
logger.info(f"Images saved to {IMAGES_DIR}")

## Part E: Export and Analyze

In [ ]:
# Load and analyze results
with open(METADATA_PATH, 'r') as f:
    metadata = json.load(f)

print(f"Generated {len(metadata)} images")
print("\nScore statistics:")
scores = [record['scores']['composite'] for record in metadata]
print(f"Average composite score: {np.mean(scores):.2f}")
print(f"Min composite score: {np.min(scores):.2f}")
print(f"Max composite score: {np.max(scores):.2f}")

# Display sample images (if in notebook environment)
try:
    from IPython.display import display
    for record in metadata[:3]:  # Show first 3
        img_path = OUTPUT_DIR / record['image_path']
        img = Image.open(img_path)
        print(f"\n{record['image_id']}: {record['subject']}")
        print(f"Composite score: {record['scores']['composite']:.2f}")
        display(img)
except ImportError:
    print("Not in notebook environment, skipping image display")

## Conclusion

This cookbook demonstrates a complete pipeline for generating high-quality synthetic datasets using:

- **DSPy** for structured prompt optimization
- **Pruna** for efficient model inference on FLUX.2-klein-4B
- **SmolVLM** for automated quality grading
- **Top-K selection** for curated output

The result is a reproducible, automated system that produces professional-quality images with explicit quality signals.

### Next Steps

- Fine-tune on the generated dataset
- Experiment with different scoring weights
- Add more sophisticated prompt optimization with DSPy
- Scale to larger subject sets